# Chintu LoRA Fine-Tuning

This notebook fine-tunes a small LLM on approved Chintu interactions using QLoRA.

**Requirements:**
- GPU runtime (T4 or better)
- Gold data exported from `chintu/training/gold_data.py`

In [ ]:
# Install dependencies
!pip install -q transformers peft bitsandbytes accelerate datasets trl

In [ ]:
# Upload your gold_interactions.jsonl file
from google.colab import files
uploaded = files.upload()  # Upload gold_interactions.jsonl

In [ ]:
# Load the dataset
from datasets import load_dataset

dataset = load_dataset('json', data_files='gold_interactions.jsonl', split='train')
print(f"Loaded {len(dataset)} interactions")
print(dataset[0])

In [ ]:
# Format for instruction tuning
def format_prompt(example):
    return {
        "text": f"""### Instruction:
{example['user_input']}

### Response:
{example['assistant_response']}"""
    }

dataset = dataset.map(format_prompt)
print(dataset[0]['text'])

In [ ]:
# Load base model with 4-bit quantization
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small model for weak hardware

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Configure LoRA
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Train with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./chintu-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    warmup_ratio=0.05,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()

In [ ]:
# Save LoRA adapter
model.save_pretrained("chintu-lora-adapter")
tokenizer.save_pretrained("chintu-lora-adapter")

# Download for local use
!zip -r chintu-lora-adapter.zip chintu-lora-adapter/
files.download("chintu-lora-adapter.zip")

In [ ]:
# Test the fine-tuned model
from peft import PeftModel

prompt = "What time is it?"
inputs = tokenizer(f"### Instruction:\n{prompt}\n\n### Response:\n", return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.7)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))